In [4]:
%pip install --upgrade scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [5]:
from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "storage-analytics-lab"

client = bigquery.Client(project=PROJECT_ID)

query = """
SELECT *
FROM `storage-analytics-lab.storage_ml_dev.facility_forecasting_features`
ORDER BY prediction_date, facility_id
"""

df = client.query(query).to_dataframe()

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Dataset shape: (1032, 11)

Columns:
['facility_id', 'prediction_date', 'day_of_week', 'month', 'bookings_today', 'bookings_lag_1', 'bookings_lag_7', 'bookings_lag_14', 'bookings_rolling_7', 'bookings_rolling_28', 'bookings_next_7_days']


,facility_id,prediction_date,day_of_week,month,bookings_today,bookings_lag_1,bookings_lag_7,bookings_lag_14,bookings_rolling_7,bookings_rolling_28,bookings_next_7_days
0,101,2025-01-15,4,1,1,8,3,1,3.714286,3.466667,15
1,102,2025-01-15,4,1,4,3,8,4,3.857143,4.600000,23
2,103,2025-01-15,4,1,7,0,3,6,4.142857,3.466667,32
3,101,2025-01-16,5,1,2,1,3,7,3.571429,3.375000,20
4,102,2025-01-16,5,1,0,4,5,1,3.142857,4.312500,27


In [6]:
df["prediction_date"] = pd.to_datetime(df["prediction_date"])

print(df.dtypes)

facility_id                     Int64
prediction_date         datetime64[s]
day_of_week                     Int64
month                           Int64
bookings_today                  Int64
bookings_lag_1                  Int64
bookings_lag_7                  Int64
bookings_lag_14                 Int64
bookings_rolling_7            float64
bookings_rolling_28           float64
bookings_next_7_days            Int64
dtype: object


In [7]:
train = df[
    df["prediction_date"] <= "2025-08-24"
].copy()

validation = df[
    (df["prediction_date"] >= "2025-09-01")
    & (df["prediction_date"] <= "2025-10-24")
].copy()

test = df[
    df["prediction_date"] >= "2025-11-01"
].copy()

for name, dataset in [
    ("Training", train),
    ("Validation", validation),
    ("Test", test),
]:
    print(
        f"{name}: {len(dataset)} observations, "
        f"{dataset['prediction_date'].min().date()} "
        f"to {dataset['prediction_date'].max().date()}"
    )

Training: 666 observations, 2025-01-15 to 2025-08-24
Validation: 162 observations, 2025-09-01 to 2025-10-24
Test: 162 observations, 2025-11-01 to 2025-12-24


In [8]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

TARGET = "bookings_next_7_days"

def evaluate_forecast(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    denominator = np.abs(y_true).sum()
    wmape = (
        np.abs(y_true - y_pred).sum() / denominator
        if denominator > 0
        else np.nan
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "WMAPE": wmape,
    }

validation_baseline = 7 * validation["bookings_rolling_7"]
test_baseline = 7 * test["bookings_rolling_7"]

baseline_results = pd.DataFrame(
    {
        "Validation": evaluate_forecast(
            validation[TARGET],
            validation_baseline
        ),
        "Test": evaluate_forecast(
            test[TARGET],
            test_baseline
        ),
    }
)

baseline_results

,Validation,Test
MAE,6.172840,8.438272
RMSE,7.740386,10.402101
WMAPE,0.242072,0.300572


In [9]:
%pip install lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 48.3 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [10]:
import lightgbm as lgb

print("LightGBM version:", lgb.__version__)

LightGBM version: 4.7.0


In [12]:
FEATURES = [
    "facility_id",
    "day_of_week",
    "month",
    "bookings_today",
    "bookings_lag_1",
    "bookings_lag_7",
    "bookings_lag_14",
    "bookings_rolling_7",
    "bookings_rolling_28",
]

X_train = train[FEATURES].copy()
y_train = train[TARGET].copy()

X_val = validation[FEATURES].copy()
y_val = validation[TARGET].copy()

X_train["facility_id"] = X_train["facility_id"].astype("category")
X_val["facility_id"] = X_val["facility_id"].astype("category")

In [15]:
model = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=100,
    learning_rate=0.05,
    num_leaves=7,
    max_depth=3,
    min_child_samples=20,
    random_state=42,
    verbosity=-1,
)

model.fit(
    X_train,
    y_train,
    categorical_feature=["facility_id"],
)

val_predictions = model.predict(X_val)

lightgbm_metrics = evaluate_forecast(
    y_val,
    val_predictions
)

comparison = pd.DataFrame(
    {
        "Baseline": baseline_results["Validation"],
        "LightGBM": pd.Series(lightgbm_metrics),
    }
)

comparison["Improvement (%)"] = (
    (comparison["Baseline"] - comparison["LightGBM"])
    / comparison["Baseline"]
    * 100
)

comparison

,Baseline,LightGBM,Improvement (%)
MAE,6.172840,6.488458,-5.113013
RMSE,7.740386,8.562874,-10.625930
WMAPE,0.242072,0.254449,-5.113013


In [16]:
train_predictions = model.predict(X_train)

train_baseline = 7 * train["bookings_rolling_7"]

diagnostics = pd.DataFrame({
    "Baseline - Train": evaluate_forecast(
        y_train, train_baseline
    ),
    "LightGBM - Train": evaluate_forecast(
        y_train, train_predictions
    ),
    "Baseline - Validation": evaluate_forecast(
        y_val, validation_baseline
    ),
    "LightGBM - Validation": evaluate_forecast(
        y_val, val_predictions
    ),
})

diagnostics

,Baseline - Train,LightGBM - Train,Baseline - Validation,LightGBM - Validation
MAE,7.271772,4.047896,6.172840,6.488458
RMSE,9.172111,5.190063,7.740386,8.562874
WMAPE,0.252424,0.140514,0.242072,0.254449


In [17]:
importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": model.feature_importances_,
})

importance.sort_values(
    "importance",
    ascending=False
)

,feature,importance
2,month,173
8,bookings_rolling_28,123
7,bookings_rolling_7,88
0,facility_id,66
5,bookings_lag_7,36
4,bookings_lag_1,23
3,bookings_today,19
6,bookings_lag_14,16
1,day_of_week,11


In [20]:
## Train Model without Month

In [21]:
FEATURES_NO_MONTH = [
    feature
    for feature in FEATURES
    if feature != "month"
]

X_train_no_month = X_train[FEATURES_NO_MONTH].copy()
X_val_no_month = X_val[FEATURES_NO_MONTH].copy()

model_no_month = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=100,
    learning_rate=0.05,
    num_leaves=7,
    max_depth=3,
    min_child_samples=20,
    random_state=42,
    verbosity=-1,
)

model_no_month.fit(
    X_train_no_month,
    y_train,
    categorical_feature=["facility_id"],
)

train_predictions_no_month = model_no_month.predict(
    X_train_no_month
)

val_predictions_no_month = model_no_month.predict(
    X_val_no_month
)

In [22]:
experiment_results = pd.DataFrame({
    "Baseline": evaluate_forecast(
        y_val,
        validation_baseline
    ),
    "LightGBM - Original": evaluate_forecast(
        y_val,
        val_predictions
    ),
    "LightGBM - No Month": evaluate_forecast(
        y_val,
        val_predictions_no_month
    ),
})

experiment_results

,Baseline,LightGBM - Original,LightGBM - No Month
MAE,6.172840,6.488458,6.136859
RMSE,7.740386,8.562874,7.945391
WMAPE,0.242072,0.254449,0.240661


In [23]:
train_comparison = pd.DataFrame({
    "Baseline": evaluate_forecast(
        y_train,
        train_baseline
    ),
    "LightGBM - Original": evaluate_forecast(
        y_train,
        train_predictions
    ),
    "LightGBM - No Month": evaluate_forecast(
        y_train,
        train_predictions_no_month
    ),
})

train_comparison

,Baseline,LightGBM - Original,LightGBM - No Month
MAE,7.271772,4.047896,4.340327
RMSE,9.172111,5.190063,5.582080
WMAPE,0.252424,0.140514,0.150665
